# Parallel merge (chapter 13)

Based on chapter 13 of *Programming Massively Parallel Processors* (Hwu, Kirk, El Hajj). The earlier patterns of the course (map, stencil, histogram, reduction, scan) have one thing in common: thread `t` knows from its index alone which input elements it must read. Merge breaks that rule. Where thread `t` should start reading depends on the **values** in the data, so each thread first has to *search* for its input. The tool for that search, the **co-rank function**, is the main idea of the chapter, and chapter 14 uses it again to build a parallel merge sort.

This notebook runs the **complete programs**: every kernel of the chapter on large inputs, compared side by side with timings, and the **bike-sharing examples**, which apply the same pattern to another domain to give you ideas for your project. Do the step-by-step notebook (`colab_1_step_by_step.ipynb`) first. Timings are only meaningful on the GPU.

The full theory, the exercises and their solutions are in `ec2/11_merge/README.md`. The slides of the session are the companion of this notebook.

### How this notebook works

Every program comes as a pair of cells:

1. a cell that starts with `%%writefile name.cu`: running it **saves** the CUDA program as a file on the
   Colab machine (it does not run anything yet);
2. a cell that starts with `!nvcc`: it **compiles** that file with NVIDIA's compiler and **runs** it on
   the GPU. The `!` sends the line to the machine's shell.

Every program computes its result on the GPU **and** with a simple loop on the CPU, compares the two and
prints **PASS** when they agree (**FAIL** otherwise). Read the code cell before running it, then read
the output: the Markdown after each run says what to look at.

You can edit any `%%writefile` cell (change a size, a constant, a kernel) and run the pair again.

## Setup: get a GPU

A new Colab notebook runs on a machine **without a GPU**. "Enabling the GPU" is not something the code
does: it means asking Colab for a machine that has an NVIDIA GPU, with the NVIDIA driver and the CUDA
toolkit (the `nvcc` compiler) already installed.

**Do this once, before running anything:** *Runtime → Change runtime type → Hardware accelerator: T4 GPU
→ Save*. Colab restarts the machine; files written before the change are lost, so run the notebook from
the top afterwards.

The next cell checks it worked. `nvidia-smi` lists the GPU (usually a *Tesla T4*) and `nvcc --version`
prints the CUDA compiler version. If you see *command not found* or *No devices were found*, the runtime
has no GPU: change it as above.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

## The `cuda_compat.h` header: what it is and when you need it

Every program in this course starts with `#include "cuda_compat.h"`. The header exists so that **the
same `.cu` file runs everywhere**, with or without an NVIDIA GPU:

| Where you run | Compiler | What `cuda_compat.h` does |
|---|---|---|
| Google Colab with a T4 GPU (this notebook) | `nvcc` | Almost nothing: it includes the real CUDA runtime, so the code runs on the real GPU. |
| Your own computer **with** an NVIDIA GPU | `nvcc` | Exactly the same as on Colab. |
| A laptop **without** an NVIDIA GPU (any Mac, most laptops) | `g++` or `clang++` | It **emulates** CUDA on the CPU: each block's threads are real CPU threads, `__syncthreads()` is a barrier, `__shared__` memory is shared, atomics and warp shuffles work. Results are correct; times are meaningless. |

It also gives three small helpers that the programs use with `nvcc` too:

- `launch(kernel, grid, block, args...)`: with `nvcc` it is **exactly** `kernel<<<grid, block>>>(args...)`.
  The `<<<...>>>` syntax only exists in `nvcc`, so a file that used it could not be compiled with `g++`;
  `launch` keeps one source for both.
- `CHECK(call)`: stops with the error message, file and line if a CUDA call fails (introduced in
  `01_first_kernels`, step 8).
- `cdiv(a, b)`: the ceiling division used to compute how many blocks a grid needs.

**On a computer with an NVIDIA GPU there is nothing to enable.** Install the NVIDIA driver and the CUDA
Toolkit (`nvcc`), then compile exactly as the cells below do:
`nvcc -O2 -arch=native -I../common file.cu -o file && ./file` (`-arch=native` compiles for the GPU in
that machine). You still include `cuda_compat.h`, because the programs call `launch` and `CHECK`; under
`nvcc` it does not emulate anything. If you prefer plain CUDA, write `kernel<<<grid, block>>>(args...)`
instead of `launch(...)` and the header is no longer needed for the launch.

**Without an NVIDIA GPU**, compile the same file with
`g++ -std=c++17 -O2 -pthread -x c++ -I../common file.cu -o file && ./file`.

The next cell saves the header on the Colab machine. You do not need to read it: just run it.

In [ ]:
%%writefile cuda_compat.h
#pragma once

#include <chrono>
#include <cstdio>
#include <cstdlib>

#ifdef __CUDACC__

#include <cuda_runtime.h>
#include <cuda/atomic>

#define CUDA_BACKEND "GPU (nvcc)"
#define BY_BACKEND(gpu, cpu) (gpu)

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  kernel<<<grid, block>>>(args...);
}

#else

#include <atomic>
#include <cstring>
#include <memory>
#include <thread>
#include <type_traits>
#include <vector>

#define CUDA_BACKEND "CPU emulation (threads)"
#define BY_BACKEND(gpu, cpu) (cpu)

#define __global__
#define __device__
#define __host__
#define __constant__
#define __shared__ static
#define __restrict__

struct dim3 {
  unsigned x;
  unsigned y;
  unsigned z;

  dim3(unsigned x_ = 1, unsigned y_ = 1, unsigned z_ = 1) : x(x_), y(y_), z(z_) {
  }
};

struct uint3 {
  unsigned x;
  unsigned y;
  unsigned z;
};

struct float4 {
  float x;
  float y;
  float z;
  float w;
};

inline thread_local uint3 threadIdx{0, 0, 0};
inline thread_local uint3 blockIdx{0, 0, 0};
inline dim3 blockDim;
inline dim3 gridDim;

constexpr int warpSize = 32;

namespace emu {

constexpr unsigned kWarpSize = warpSize;
constexpr unsigned kFullWarpMask = 0xffffffffu;
constexpr int kCudaAtomicFunctionOrder = __ATOMIC_RELAXED;

class Barrier {
 public:
  explicit Barrier(unsigned threadCount) : threadCount_(threadCount) {
  }

  void arriveAndWait() {
    const unsigned generationOnArrival = generation_.load(std::memory_order_acquire);
    const bool lastToArrive = arrived_.fetch_add(1, std::memory_order_acq_rel) + 1 == threadCount_;

    if (lastToArrive) {
      releaseEveryWaitingThread();
    } else {
      yieldUntilGenerationEnds(generationOnArrival);
    }
  }

 private:
  void releaseEveryWaitingThread() {
    arrived_.store(0, std::memory_order_relaxed);
    generation_.fetch_add(1, std::memory_order_acq_rel);
  }

  void yieldUntilGenerationEnds(unsigned generation) const {
    while (generation_.load(std::memory_order_acquire) == generation) {
      std::this_thread::yield();
    }
  }

  std::atomic<unsigned> arrived_{0};
  std::atomic<unsigned> generation_{0};
  const unsigned threadCount_;
};

inline Barrier* blockBarrier = nullptr;

using LaneBits = unsigned long long;

class Warp {
 public:
  explicit Warp(unsigned laneCount) : laneCount_(laneCount), barrier_(laneCount), published_{} {
  }

  unsigned laneCount() const {
    return laneCount_;
  }

  unsigned everyLaneMask() const {
    if (laneCount_ == kWarpSize) {
      return kFullWarpMask;
    }
    return (1u << laneCount_) - 1u;
  }

  bool hasLane(int lane) const {
    return lane >= 0 && lane < static_cast<int>(laneCount_);
  }

  void publish(unsigned lane, LaneBits bits) {
    __atomic_store_n(&published_[lane], bits, __ATOMIC_RELAXED);
  }

  LaneBits publishedBy(int lane) const {
    return __atomic_load_n(&published_[lane], __ATOMIC_RELAXED);
  }

  void waitForEveryLane() {
    barrier_.arriveAndWait();
  }

 private:
  const unsigned laneCount_;
  Barrier barrier_;
  LaneBits published_[kWarpSize];
};

inline thread_local Warp* warpOfThisThread = nullptr;
inline thread_local unsigned laneOfThisThread = 0;

template <class T>
LaneBits toLaneBits(T value) {
  static_assert(sizeof(T) <= sizeof(LaneBits), "shuffle of a type wider than 8 bytes");
  LaneBits bits = 0;
  std::memcpy(&bits, &value, sizeof(T));
  return bits;
}

template <class T>
T fromLaneBits(LaneBits bits) {
  T value;
  std::memcpy(&value, &bits, sizeof(T));
  return value;
}

template <class T>
T readValueOfLane(T myValue, int sourceLane) {
  Warp& myWarp = *warpOfThisThread;
  myWarp.publish(laneOfThisThread, toLaneBits(myValue));
  myWarp.waitForEveryLane();

  T received = myValue;
  if (myWarp.hasLane(sourceLane)) {
    received = fromLaneBits<T>(myWarp.publishedBy(sourceLane));
  }

  myWarp.waitForEveryLane();
  return received;
}

inline int positionInGroup(int lane, int width) {
  return lane % width;
}

inline int firstLaneOfGroup(int lane, int width) {
  return lane / width * width;
}

inline bool inSameGroup(int lane, int otherLane, int width) {
  return lane / width == otherLane / width;
}

inline int shflDownSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) + delta < width;
  return sourceInGroup ? lane + delta : lane;
}

inline int shflUpSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) >= delta;
  return sourceInGroup ? lane - delta : lane;
}

inline int shflXorSourceLane(int lane, int laneMask, int width) {
  const int partner = lane ^ laneMask;
  return inSameGroup(lane, partner, width) ? partner : lane;
}

inline int shflSourceLane(int lane, int srcLane, int width) {
  const int wrappedSrcLane = (srcLane % width + width) % width;
  return firstLaneOfGroup(lane, width) + wrappedSrcLane;
}

inline int myLane() {
  return static_cast<int>(laneOfThisThread);
}

}

inline void __syncthreads() {
  emu::blockBarrier->arriveAndWait();
}

inline void __syncwarp([[maybe_unused]] unsigned mask = emu::kFullWarpMask) {
  emu::warpOfThisThread->waitForEveryLane();
}

template <class T>
T __shfl_down_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflDownSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_up_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflUpSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_xor_sync([[maybe_unused]] unsigned mask, T var, int laneMask, int width = warpSize) {
  const int sourceLane = emu::shflXorSourceLane(emu::myLane(), laneMask, width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_sync([[maybe_unused]] unsigned mask, T var, int srcLane, int width = warpSize) {
  const int sourceLane = emu::shflSourceLane(emu::myLane(), srcLane, width);
  return emu::readValueOfLane(var, sourceLane);
}

inline unsigned __ballot_sync([[maybe_unused]] unsigned mask, int predicate) {
  unsigned ballot = 0;
  const unsigned laneCount = emu::warpOfThisThread->laneCount();

  for (unsigned sourceLane = 0; sourceLane < laneCount; ++sourceLane) {
    const int predicateOfLane =
        __shfl_sync(emu::kFullWarpMask, predicate, static_cast<int>(sourceLane));
    if (predicateOfLane) {
      ballot |= 1u << sourceLane;
    }
  }

  return ballot;
}

inline int __any_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) != 0;
}

inline int __all_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) == emu::warpOfThisThread->everyLaneMask();
}

namespace emu {

using WarpList = std::vector<std::unique_ptr<Warp>>;

inline unsigned threadCountOf(dim3 block) {
  return block.x * block.y * block.z;
}

inline WarpList splitIntoWarps(unsigned threadCount) {
  WarpList warps;

  for (unsigned firstThread = 0; firstThread < threadCount; firstThread += kWarpSize) {
    const unsigned threadsLeft = threadCount - firstThread;
    const unsigned laneCount = threadsLeft < kWarpSize ? threadsLeft : kWarpSize;
    warps.push_back(std::make_unique<Warp>(laneCount));
  }

  return warps;
}

inline uint3 threadIndexOf(unsigned linearThread, dim3 block) {
  const unsigned x = linearThread % block.x;
  const unsigned y = linearThread / block.x % block.y;
  const unsigned z = linearThread / (block.x * block.y);
  return {x, y, z};
}

inline void waitForTheWholeBlockToFinish() {
  blockBarrier->arriveAndWait();
}

template <class Kernel, class... Args>
void runThreadInEveryBlock(
    unsigned linearThread, Warp* threadWarp, dim3 grid, dim3 block, Kernel kernel, Args... args) {
  warpOfThisThread = threadWarp;
  laneOfThisThread = linearThread % kWarpSize;
  threadIdx = threadIndexOf(linearThread, block);

  for (unsigned bz = 0; bz < grid.z; ++bz) {
    for (unsigned by = 0; by < grid.y; ++by) {
      for (unsigned bx = 0; bx < grid.x; ++bx) {
        blockIdx = {bx, by, bz};
        kernel(args...);
        waitForTheWholeBlockToFinish();
      }
    }
  }
}

constexpr int kErrorInvalidConfiguration = 9;
constexpr unsigned kMaxThreadsPerBlock = 1024;
constexpr unsigned kMaxBlockDimZ = 64;
constexpr unsigned kMaxGridDimYZ = 65535;

inline int lastLaunchError = 0;

inline bool isValidLaunchConfiguration(dim3 grid, dim3 block) {
  const bool blockFits = block.x >= 1 && block.y >= 1 && block.z >= 1 &&
                         block.x <= kMaxThreadsPerBlock && block.y <= kMaxThreadsPerBlock &&
                         block.z <= kMaxBlockDimZ &&
                         threadCountOf(block) <= kMaxThreadsPerBlock;
  const bool gridFits = grid.x >= 1 && grid.y >= 1 && grid.z >= 1 && grid.y <= kMaxGridDimYZ &&
                        grid.z <= kMaxGridDimYZ;
  return blockFits && gridFits;
}

}

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  if (!emu::isValidLaunchConfiguration(grid, block)) {
    emu::lastLaunchError = emu::kErrorInvalidConfiguration;
    return;
  }

  gridDim = grid;
  blockDim = block;

  const unsigned threadCount = emu::threadCountOf(block);
  emu::Barrier barrier(threadCount);
  emu::blockBarrier = &barrier;
  emu::WarpList warps = emu::splitIntoWarps(threadCount);

  std::vector<std::thread> threads;
  threads.reserve(threadCount);
  for (unsigned linearThread = 0; linearThread < threadCount; ++linearThread) {
    emu::Warp* threadWarp = warps[linearThread / emu::kWarpSize].get();
    threads.emplace_back([=] {
      emu::runThreadInEveryBlock(linearThread, threadWarp, grid, block, kernel, args...);
    });
  }

  for (std::thread& thread : threads) {
    thread.join();
  }
  emu::blockBarrier = nullptr;
}

namespace emu {

template <class T, class Improves>
T atomicReplaceWhileImproving(T* address, T candidate, Improves improves) {
  T current = __atomic_load_n(address, kCudaAtomicFunctionOrder);

  while (improves(candidate, current)) {
    const bool replaced = __atomic_compare_exchange_n(
        address, &current, candidate, true, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
    if (replaced) {
      break;
    }
  }

  return current;
}

template <class T>
T atomicStoreMinimum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate < current;
  });
}

template <class T>
T atomicStoreMaximum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate > current;
  });
}

template <class T>
T atomicCompareAndSwap(T* address, T compare, T value) {
  T observed = compare;
  __atomic_compare_exchange_n(
      address, &observed, value, false, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
  return observed;
}

}

inline int atomicAdd(int* address, int value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned atomicAdd(unsigned* address, unsigned value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned long long atomicAdd(unsigned long long* address, unsigned long long value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline float atomicAdd(float* address, float value) {
  float old;
  __atomic_load(address, &old, emu::kCudaAtomicFunctionOrder);
  float sum = old + value;

  while (!__atomic_compare_exchange(
      address, &old, &sum, true, emu::kCudaAtomicFunctionOrder, emu::kCudaAtomicFunctionOrder)) {
    sum = old + value;
  }

  return old;
}

inline int atomicMin(int* address, int value) {
  return emu::atomicStoreMinimum(address, value);
}

inline int atomicMax(int* address, int value) {
  return emu::atomicStoreMaximum(address, value);
}

inline unsigned long long atomicMin(unsigned long long* address, unsigned long long value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMin(unsigned* address, unsigned value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMax(unsigned* address, unsigned value) {
  return emu::atomicStoreMaximum(address, value);
}

inline int atomicSub(int* address, int value) {
  return __atomic_fetch_sub(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicExch(int* address, int value) {
  return __atomic_exchange_n(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicCAS(int* address, int compare, int value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline unsigned long long atomicCAS(unsigned long long* address,
                                    unsigned long long compare,
                                    unsigned long long value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline void __threadfence() {
  __atomic_thread_fence(__ATOMIC_SEQ_CST);
}

namespace cuda {

enum thread_scope {
  thread_scope_system = 0,
  thread_scope_device = 1,
  thread_scope_block = 2,
  thread_scope_thread = 3
};

namespace std {

enum memory_order {
  memory_order_relaxed = __ATOMIC_RELAXED,
  memory_order_consume = __ATOMIC_CONSUME,
  memory_order_acquire = __ATOMIC_ACQUIRE,
  memory_order_release = __ATOMIC_RELEASE,
  memory_order_acq_rel = __ATOMIC_ACQ_REL,
  memory_order_seq_cst = __ATOMIC_SEQ_CST
};

}

using std::memory_order;
using std::memory_order_acq_rel;
using std::memory_order_acquire;
using std::memory_order_relaxed;
using std::memory_order_release;
using std::memory_order_seq_cst;

template <class T, thread_scope Scope = thread_scope_system>
class atomic_ref {
 public:
  explicit atomic_ref(T& object) : address_(&object) {
  }

  T load(memory_order order = memory_order_seq_cst) const {
    T value;
    __atomic_load(address_, &value, order);
    return value;
  }

  void store(T value, memory_order order = memory_order_seq_cst) const {
    __atomic_store(address_, &value, order);
  }

  T exchange(T value, memory_order order = memory_order_seq_cst) const {
    T old;
    __atomic_exchange(address_, &value, &old, order);
    return old;
  }

  bool compare_exchange_strong(T& expected,
                               T desired,
                               memory_order order = memory_order_seq_cst) const {
    return __atomic_compare_exchange(address_, &expected, &desired, false, order, __ATOMIC_RELAXED);
  }

  T fetch_add(T value, memory_order order = memory_order_seq_cst) const {
    if constexpr (::std::is_integral_v<T>) {
      return __atomic_fetch_add(address_, value, order);
    } else {
      return fetchAddByCompareExchange(value, order);
    }
  }

  T fetch_sub(T value, memory_order order = memory_order_seq_cst) const {
    return fetch_add(-value, order);
  }

  T fetch_min(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate < current;
    });
  }

  T fetch_max(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate > current;
    });
  }

 private:
  T fetchAddByCompareExchange(T value, memory_order order) const {
    T old = load(memory_order_relaxed);
    bool replaced = false;
    while (!replaced) {
      replaced = compare_exchange_strong(old, old + value, order);
    }
    return old;
  }

  template <class Improves>
  T replaceWhileImproving(T candidate, memory_order order, Improves improves) const {
    T current = load(memory_order_relaxed);
    while (improves(candidate, current)) {
      if (compare_exchange_strong(current, candidate, order)) {
        break;
      }
    }
    return current;
  }

  T* address_;
};

}

enum cudaMemcpyKind {
  cudaMemcpyHostToDevice,
  cudaMemcpyDeviceToHost,
  cudaMemcpyDeviceToDevice,
  cudaMemcpyHostToHost
};

typedef int cudaError_t;

const cudaError_t cudaSuccess = 0;

const cudaError_t cudaErrorInvalidConfiguration = emu::kErrorInvalidConfiguration;

inline const char* cudaGetErrorString(cudaError_t error) {
  if (error == cudaErrorInvalidConfiguration) {
    return "invalid configuration argument";
  }
  return "no error";
}

inline cudaError_t cudaPeekAtLastError() {
  return emu::lastLaunchError;
}

inline cudaError_t cudaGetLastError() {
  const cudaError_t error = emu::lastLaunchError;
  emu::lastLaunchError = cudaSuccess;
  return error;
}

inline cudaError_t cudaDeviceSynchronize() {
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMalloc(T** devicePointer, size_t bytes) {
  *devicePointer = static_cast<T*>(std::malloc(bytes));
  return cudaSuccess;
}

inline cudaError_t cudaFree(void* devicePointer) {
  std::free(devicePointer);
  return cudaSuccess;
}

inline cudaError_t cudaMemcpy(void* destination, const void* source, size_t bytes, cudaMemcpyKind) {
  std::memcpy(destination, source, bytes);
  return cudaSuccess;
}

inline cudaError_t cudaMemset(void* destination, int byteValue, size_t bytes) {
  std::memset(destination, byteValue, bytes);
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMemcpyToSymbol(T& symbol, const void* source, size_t bytes) {
  std::memcpy(&symbol, source, bytes);
  return cudaSuccess;
}

struct cudaDeviceProp {
  char name[256];
  int major;
  int minor;
  int multiProcessorCount;
  int maxThreadsPerBlock;
  int maxThreadsDim[3];
  int maxGridSize[3];
  int maxThreadsPerMultiProcessor;
  int maxBlocksPerMultiProcessor;
  int warpSize;
  int regsPerBlock;
  int regsPerMultiprocessor;
  size_t sharedMemPerBlock;
  size_t sharedMemPerMultiprocessor;
  size_t totalGlobalMem;
  size_t totalConstMem;
  int clockRate;
};

inline cudaError_t cudaGetDeviceCount(int* deviceCount) {
  *deviceCount = 1;
  return cudaSuccess;
}

inline cudaError_t cudaGetDeviceProperties(cudaDeviceProp* properties, int) {
  constexpr int MAX_THREADS_PER_BLOCK = 1024;
  constexpr int MAX_GRID_X = 2147483647;
  constexpr int MAX_GRID_YZ = 65535;
  constexpr int MAX_BLOCK_Z = 64;
  constexpr size_t KIB = 1024;
  *properties = cudaDeviceProp{};
  std::snprintf(properties->name, sizeof properties->name, "CPU emulation (cuda_compat.h)");
  properties->multiProcessorCount = static_cast<int>(std::thread::hardware_concurrency());
  properties->maxThreadsPerBlock = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[0] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[1] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[2] = MAX_BLOCK_Z;
  properties->maxGridSize[0] = MAX_GRID_X;
  properties->maxGridSize[1] = MAX_GRID_YZ;
  properties->maxGridSize[2] = MAX_GRID_YZ;
  properties->maxThreadsPerMultiProcessor = 2 * MAX_THREADS_PER_BLOCK;
  properties->maxBlocksPerMultiProcessor = 32;
  properties->warpSize = warpSize;
  properties->regsPerBlock = 64 * 1024;
  properties->regsPerMultiprocessor = 64 * 1024;
  properties->sharedMemPerBlock = 48 * KIB;
  properties->sharedMemPerMultiprocessor = 100 * KIB;
  properties->totalConstMem = 64 * KIB;
  return cudaSuccess;
}

#endif

#define CHECK(call)                                                                          \
  do {                                                                                       \
    cudaError_t err_ = (call);                                                               \
    if (err_ != cudaSuccess) {                                                               \
      std::fprintf(                                                                          \
          stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_), __FILE__, __LINE__); \
      std::exit(1);                                                                          \
    }                                                                                        \
  } while (0)

struct Timer {
  std::chrono::steady_clock::time_point t0 = std::chrono::steady_clock::now();

  double ms() {
    cudaDeviceSynchronize();
    const auto elapsed = std::chrono::steady_clock::now() - t0;
    return std::chrono::duration<double, std::milli>(elapsed).count();
  }
};

inline unsigned cdiv(unsigned a, unsigned b) {
  return (a + b - 1) / b;
}

__host__ __device__ inline int imin(int a, int b) {
  return a < b ? a : b;
}

__host__ __device__ inline int imax(int a, int b) {
  return a > b ? a : b;
}

__host__ __device__ inline float fminf_(float a, float b) {
  return a < b ? a : b;
}

__host__ __device__ inline float fmaxf_(float a, float b) {
  return a > b ? a : b;
}


## `bikeshare_station_logs.cu`

Stable tree merge of per-station dock/undock logs into one city log sorted by `t`

This is a bike-sharing example: the README section *Ideas for your project* explains it and lists where the same pattern may appear in your project.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile bikeshare_station_logs.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <vector>

enum class EventKind : int { Undock, Dock };

struct Event {
  int timeMs;
  EventKind kind;
  int stationId;
  int bikeId;
};

__host__ __device__ inline int co_rank_events(int k, const Event* A, int m, const Event* B, int n) {
  int i = imin(k, m);
  int j = k - i;
  int i_low = imax(0, k - n);
  int j_low = imax(0, k - m);
  bool searching = true;
  while (searching) {
    const bool i_too_large = i > 0 && j < n && A[i - 1].timeMs > B[j].timeMs;
    const bool i_too_small = j > 0 && i < m && B[j - 1].timeMs >= A[i].timeMs;
    if (i_too_large) {
      const int delta = (i - i_low + 1) >> 1;
      j_low = j;
      i -= delta;
      j += delta;
    } else if (i_too_small) {
      const int delta = (j - j_low + 1) >> 1;
      i_low = i;
      i += delta;
      j -= delta;
    } else {
      searching = false;
    }
  }
  return i;
}

__host__ __device__ inline void merge_events(
    const Event* A, int m, const Event* B, int n, Event* C) {
  int i = 0;
  int j = 0;
  int k = 0;
  while (i < m && j < n) {
    if (A[i].timeMs <= B[j].timeMs) {
      C[k++] = A[i++];
    } else {
      C[k++] = B[j++];
    }
  }
  while (i < m) {
    C[k++] = A[i++];
  }
  while (j < n) {
    C[k++] = B[j++];
  }
}

struct ListPair {
  int leftBegin;
  int rightBegin;
  int end;
};

__host__ __device__ inline int last_pair_starting_at_or_before(int k,
                                                               const int* listBegin,
                                                               int numPairs) {
  int low = 0;
  int high = numPairs - 1;
  while (low < high) {
    const int middle = (low + high + 1) / 2;
    if (listBegin[2 * middle] <= k) {
      low = middle;
    } else {
      high = middle - 1;
    }
  }
  return low;
}

__host__ __device__ inline ListPair list_pair(int pair, const int* listBegin, int numLists) {
  const int left = 2 * pair;
  const int rightOrEmpty = imin(left + 1, numLists);
  const int afterRight = imin(left + 2, numLists);
  return {listBegin[left], listBegin[rightOrEmpty], listBegin[afterRight]};
}

__global__ void merge_round_kernel(const Event* in,
                                   Event* out,
                                   const int* listBegin,
                                   int numLists,
                                   int total,
                                   int outputsPerThread) {
  const int tid = blockIdx.x * blockDim.x + threadIdx.x;
  const int numPairs = (numLists + 1) / 2;
  const int sliceBegin = imin(tid * outputsPerThread, total);
  const int sliceEnd = imin(sliceBegin + outputsPerThread, total);

  int k = sliceBegin;
  while (k < sliceEnd) {
    const int pairIndex = last_pair_starting_at_or_before(k, listBegin, numPairs);
    const ListPair pair = list_pair(pairIndex, listBegin, numLists);
    const int pieceEnd = imin(sliceEnd, pair.end);

    const Event* A = in + pair.leftBegin;
    const Event* B = in + pair.rightBegin;
    const int m = pair.rightBegin - pair.leftBegin;
    const int n = pair.end - pair.rightBegin;
    const int k_curr = k - pair.leftBegin;
    const int k_next = pieceEnd - pair.leftBegin;
    const int i_curr = co_rank_events(k_curr, A, m, B, n);
    const int i_next = co_rank_events(k_next, A, m, B, n);
    const int j_curr = k_curr - i_curr;
    const int j_next = k_next - i_next;
    merge_events(A + i_curr, i_next - i_curr, B + j_curr, j_next - j_curr, out + k);

    k = pieceEnd;
  }
}

#ifndef CPU_BIKES
#define CPU_BIKES 240
#endif
#ifndef CPU_TRIPS_PER_BIKE
#define CPU_TRIPS_PER_BIKE 150
#endif
constexpr int STATIONS = 48;
constexpr int BIKES = BY_BACKEND(10000, CPU_BIKES);
constexpr int TRIPS_PER_BIKE = BY_BACKEND(400, CPU_TRIPS_PER_BIKE);
constexpr uint64_t LOG_SEED = 17;

constexpr int STATION_CLOCK_TICK_MS = 1000;
constexpr int SECONDS_PER_MINUTE = 60;
constexpr int PARKED_TICKS_MAX = 45 * SECONDS_PER_MINUTE;
constexpr int RIDE_TICKS_MIN = 3 * SECONDS_PER_MINUTE;
constexpr int RIDE_TICKS_MAX = 25 * SECONDS_PER_MINUTE;

constexpr int OUTPUTS_PER_THREAD = 32;
constexpr int THREADS_PER_BLOCK = 256;

constexpr int FIRST_EVENTS_SHOWN = 6;
constexpr int TIE_GROUP_MIN_SIZE = 3;
constexpr int TIE_GROUP_MAX_SHOWN = 4;

constexpr int MS_PER_SECOND = 1000;
constexpr int SECONDS_PER_HOUR = 60 * SECONDS_PER_MINUTE;
constexpr int HOURS_PER_DAY = 24;

using StationLog = std::vector<Event>;

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_RATIO_INCREMENT;
    uint64_t z = state_;
    z = (z ^ (z >> 30)) * FIRST_MIX_MULTIPLIER;
    z = (z ^ (z >> 27)) * SECOND_MIX_MULTIPLIER;
    return z ^ (z >> 31);
  }

  int between(int lowest, int highest) {
    const uint64_t choices = static_cast<uint64_t>(highest - lowest) + 1;
    return lowest + static_cast<int>(next() % choices);
  }

 private:
  static constexpr uint64_t GOLDEN_RATIO_INCREMENT = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MIX_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MIX_MULTIPLIER = 0x94D049BB133111EBull;

  uint64_t state_;
};

static const char* eventKindName(EventKind kind) {
  switch (kind) {
    case EventKind::Undock:
      return "undock";
    case EventKind::Dock:
      return "dock";
  }
  return "unknown";
}

static bool earlier(const Event& a, const Event& b) {
  return a.timeMs < b.timeMs;
}

class SyntheticCityWeek {
 public:
  explicit SyntheticCityWeek(uint64_t seed) : rng_(seed) {
  }

  std::vector<StationLog> makeLogs(int bikes, int tripsPerBike) {
    std::vector<StationLog> logs(STATIONS);
    for (int bike = 0; bike < bikes; ++bike) {
      recordTripsOfOneBike(bike, tripsPerBike, logs);
    }
    for (StationLog& log : logs) {
      std::stable_sort(log.begin(), log.end(), earlier);
    }
    return logs;
  }

 private:
  void recordTripsOfOneBike(int bike, int tripsPerBike, std::vector<StationLog>& logs) {
    int station = randomStation();
    int timeMs = ticks(0, PARKED_TICKS_MAX);
    for (int trip = 0; trip < tripsPerBike; ++trip) {
      const int endStation = randomStation();
      logs[station].push_back({timeMs, EventKind::Undock, station, bike});
      timeMs += ticks(RIDE_TICKS_MIN, RIDE_TICKS_MAX);
      logs[endStation].push_back({timeMs, EventKind::Dock, endStation, bike});
      timeMs += ticks(0, PARKED_TICKS_MAX);
      station = endStation;
    }
  }

  int randomStation() {
    return rng_.between(0, STATIONS - 1);
  }

  int ticks(int fewest, int most) {
    return STATION_CLOCK_TICK_MS * rng_.between(fewest, most);
  }

  SplitMix64 rng_;
};

struct ConcatenatedLogs {
  std::vector<Event> events;
  std::vector<int> listBegin;
};

static ConcatenatedLogs concatenate(const std::vector<StationLog>& logs) {
  ConcatenatedLogs result;
  result.listBegin.push_back(0);
  for (const StationLog& log : logs) {
    result.events.insert(result.events.end(), log.begin(), log.end());
    result.listBegin.push_back(static_cast<int>(result.events.size()));
  }
  return result;
}

static std::vector<int> listBeginAfterRound(const std::vector<int>& listBegin, int numLists) {
  std::vector<int> merged;
  for (int left = 0; left < numLists; left += 2) {
    merged.push_back(listBegin[left]);
  }
  merged.push_back(listBegin[numLists]);
  return merged;
}

static int otherBuffer(int buffer) {
  return 1 - buffer;
}

struct TreeMergeResult {
  std::vector<Event> merged;
  int rounds;
  double ms;
};

static TreeMergeResult mergeTreeOnDevice(const ConcatenatedLogs& logs) {
  const int total = static_cast<int>(logs.events.size());
  const size_t eventBytes = total * sizeof(Event);
  Event* deviceBuffers[2];
  int* deviceListBegin;
  CHECK(cudaMalloc(&deviceBuffers[0], eventBytes));
  CHECK(cudaMalloc(&deviceBuffers[1], eventBytes));
  CHECK(cudaMalloc(&deviceListBegin, logs.listBegin.size() * sizeof(int)));
  CHECK(cudaMemcpy(deviceBuffers[0], logs.events.data(), eventBytes, cudaMemcpyHostToDevice));

  const unsigned threadsNeeded = cdiv(total, OUTPUTS_PER_THREAD);
  const unsigned blocks = cdiv(threadsNeeded, THREADS_PER_BLOCK);
  std::vector<int> listBegin = logs.listBegin;
  int numLists = static_cast<int>(logs.listBegin.size()) - 1;
  int source = 0;
  int rounds = 0;

  Timer timer;
  while (numLists > 1) {
    CHECK(cudaMemcpy(
        deviceListBegin, listBegin.data(), (numLists + 1) * sizeof(int), cudaMemcpyHostToDevice));
    launch(merge_round_kernel,
           dim3(blocks),
           dim3(THREADS_PER_BLOCK),
           deviceBuffers[source],
           deviceBuffers[otherBuffer(source)],
           deviceListBegin,
           numLists,
           total,
           OUTPUTS_PER_THREAD);
    CHECK(cudaGetLastError());

    listBegin = listBeginAfterRound(listBegin, numLists);
    numLists = (numLists + 1) / 2;
    source = otherBuffer(source);
    ++rounds;
  }
  const double ms = timer.ms();

  TreeMergeResult result{std::vector<Event>(total), rounds, ms};
  CHECK(
      cudaMemcpy(result.merged.data(), deviceBuffers[source], eventBytes, cudaMemcpyDeviceToHost));
  CHECK(cudaFree(deviceBuffers[0]));
  CHECK(cudaFree(deviceBuffers[1]));
  CHECK(cudaFree(deviceListBegin));
  return result;
}

static bool sameEvent(const Event& a, const Event& b) {
  return a.timeMs == b.timeMs && a.kind == b.kind && a.stationId == b.stationId &&
         a.bikeId == b.bikeId;
}

static bool sameEvents(const std::vector<Event>& a, const std::vector<Event>& b) {
  return std::equal(a.begin(), a.end(), b.begin(), b.end(), sameEvent);
}

static bool sortedByTime(const std::vector<Event>& events) {
  return std::is_sorted(events.begin(), events.end(), earlier);
}

static bool equalTimesListLowerStationFirst(const std::vector<Event>& events) {
  for (size_t i = 1; i < events.size(); ++i) {
    const Event& previous = events[i - 1];
    const Event& current = events[i];
    if (previous.timeMs == current.timeMs && previous.stationId > current.stationId) {
      return false;
    }
  }
  return true;
}

static bool eachBikeAlternatesAndStaysPut(const std::vector<Event>& events, int numBikes) {
  const int noStationYet = -1;
  std::vector<int> lastTimeMs(numBikes, 0);
  std::vector<EventKind> lastKind(numBikes, EventKind::Dock);
  std::vector<int> dockedAt(numBikes, noStationYet);
  bool ok = true;
  for (const Event& event : events) {
    const int bike = event.bikeId;
    const bool timeAdvances = event.timeMs >= lastTimeMs[bike];
    const bool kindAlternates = event.kind != lastKind[bike];
    const bool leavesWhereItWasDocked = event.kind == EventKind::Dock ||
                                        dockedAt[bike] == noStationYet ||
                                        dockedAt[bike] == event.stationId;
    ok = ok && timeAdvances && kindAlternates && leavesWhereItWasDocked;
    lastTimeMs[bike] = event.timeMs;
    lastKind[bike] = event.kind;
    if (event.kind == EventKind::Dock) {
      dockedAt[bike] = event.stationId;
    }
  }
  return ok;
}

static void printEvent(const Event& event) {
  const int totalSeconds = event.timeMs / MS_PER_SECOND;
  const int day = totalSeconds / (SECONDS_PER_HOUR * HOURS_PER_DAY);
  const int hour = totalSeconds / SECONDS_PER_HOUR % HOURS_PER_DAY;
  const int minute = totalSeconds / SECONDS_PER_MINUTE % SECONDS_PER_MINUTE;
  const int second = totalSeconds % SECONDS_PER_MINUTE;
  std::printf("  day %d %02d:%02d:%02d  station s%02d  %-6s  bike b%04d\n",
              day,
              hour,
              minute,
              second,
              event.stationId,
              eventKindName(event.kind),
              event.bikeId);
}

static void printFirstEvents(const std::vector<Event>& events) {
  std::printf("first lines of the city log:\n");
  const int shown = std::min<int>(FIRST_EVENTS_SHOWN, events.size());
  for (int i = 0; i < shown; ++i) {
    printEvent(events[i]);
  }
}

static bool startsTieGroup(const std::vector<Event>& events, size_t i) {
  if (i == 0 || i + TIE_GROUP_MIN_SIZE > events.size()) {
    return false;
  }
  const int timeMs = events[i].timeMs;
  return events[i - 1].timeMs != timeMs && events[i + 1].timeMs == timeMs &&
         events[i + 2].timeMs == timeMs;
}

static void printFirstTieGroup(const std::vector<Event>& events) {
  std::printf("a group of events in the same second (lower station ids first):\n");
  for (size_t first = 0; first < events.size(); ++first) {
    if (startsTieGroup(events, first)) {
      const size_t last = std::min(events.size(), first + TIE_GROUP_MAX_SHOWN);
      for (size_t i = first; i < last && events[i].timeMs == events[first].timeMs; ++i) {
        printEvent(events[i]);
      }
      return;
    }
  }
}

static void printTimings(double cpuMs, const TreeMergeResult& tree) {
  std::printf("\n%-40s %10s\n", "step", "time (ms)");
  std::printf("%-40s %10.2f\n", "std::stable_sort by t (CPU, reference)", cpuMs);
  std::printf("%-40s %10.2f\n", "tree of co-rank merges (device)", tree.ms);
  std::printf("rounds: %d (ceil(log2 %d))\n\n", tree.rounds, STATIONS);
}

static bool reportCheck(const char* label, bool ok) {
  std::printf("%-43s%s\n", label, ok ? "PASS" : "FAIL");
  return ok;
}

int main() {
  std::printf("Merging per-station dock logs on %s\n", CUDA_BACKEND);

  SyntheticCityWeek simulation(LOG_SEED);
  const ConcatenatedLogs logs = concatenate(simulation.makeLogs(BIKES, TRIPS_PER_BIKE));
  std::printf("%d stations, %d bikes, %d events in all\n\n",
              STATIONS,
              BIKES,
              static_cast<int>(logs.events.size()));

  std::vector<Event> reference = logs.events;
  Timer cpuTimer;
  std::stable_sort(reference.begin(), reference.end(), earlier);
  const double cpuMs = cpuTimer.ms();

  const TreeMergeResult tree = mergeTreeOnDevice(logs);
  const std::vector<Event>& merged = tree.merged;

  printFirstEvents(merged);
  printFirstTieGroup(merged);
  printTimings(cpuMs, tree);

  bool ok = true;
  ok = reportCheck("same as stable sort:", sameEvents(merged, reference)) && ok;
  ok = reportCheck("sorted by t:", sortedByTime(merged)) && ok;
  ok = reportCheck("equal t, lower station first:", equalTimesListLowerStationFirst(merged)) && ok;
  ok = reportCheck("per bike, undock/dock alternate in place:",
                   eachBikeAlternatesAndStaysPut(merged, BIKES)) &&
       ok;
  std::printf("%s\n", ok ? "ALL PASS" : "SOME TESTS FAILED");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. bikeshare_station_logs.cu -o bikeshare_station_logs && ./bikeshare_station_logs

## `merge.cu`

`co_rank`, the sequential merge, the basic, tiled and circular-buffer kernels; edge-case tests; a timing table against `std::merge`

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile merge.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cstdio>
#include <iterator>
#include <random>
#include <vector>

__host__ __device__ inline void merge_sequential(const int* A, int m, const int* B, int n, int* C) {
  int i = 0;
  int j = 0;
  int k = 0;
  while (i < m && j < n) {
    if (A[i] <= B[j]) {
      C[k++] = A[i++];
    } else {
      C[k++] = B[j++];
    }
  }
  while (i < m) {
    C[k++] = A[i++];
  }
  while (j < n) {
    C[k++] = B[j++];
  }
}

__host__ __device__ inline int co_rank(int k, const int* A, int m, const int* B, int n) {
  int i = imin(k, m);
  int j = k - i;
  int i_low = imax(0, k - n);
  int j_low = imax(0, k - m);
  bool searching = true;
  while (searching) {
    const bool i_too_large = i > 0 && j < n && A[i - 1] > B[j];
    const bool i_too_small = j > 0 && i < m && B[j - 1] >= A[i];
    if (i_too_large) {
      const int delta = (i - i_low + 1) >> 1;
      j_low = j;
      i -= delta;
      j += delta;
    } else if (i_too_small) {
      const int delta = (j - j_low + 1) >> 1;
      i_low = i;
      i += delta;
      j -= delta;
    } else {
      searching = false;
    }
  }
  return i;
}

__global__ void merge_basic_kernel(const int* A, int m, const int* B, int n, int* C) {
  const int tid = blockIdx.x * blockDim.x + threadIdx.x;
  const int totalThreads = gridDim.x * blockDim.x;
  const int elementsPerThread = (m + n + totalThreads - 1) / totalThreads;

  const int k_curr = imin(tid * elementsPerThread, m + n);
  const int k_next = imin(k_curr + elementsPerThread, m + n);
  const int i_curr = co_rank(k_curr, A, m, B, n);
  const int i_next = co_rank(k_next, A, m, B, n);
  const int j_curr = k_curr - i_curr;
  const int j_next = k_next - i_next;

  merge_sequential(A + i_curr, i_next - i_curr, B + j_curr, j_next - j_curr, C + k_curr);
}

constexpr int TILE = 1024;
constexpr int TILED_THREADS_PER_BLOCK = 128;
constexpr int TILE_OUTPUTS_PER_THREAD = TILE / TILED_THREADS_PER_BLOCK;
static_assert(TILE % TILED_THREADS_PER_BLOCK == 0, "every thread merges the same share of a tile");

__global__ void merge_tiled_kernel(const int* A, int m, const int* B, int n, int* C) {
  __shared__ int A_S[TILE];
  __shared__ int B_S[TILE];
  __shared__ int block_A_curr;
  __shared__ int block_A_next;

  const int outputsPerBlock = (m + n + gridDim.x - 1) / gridDim.x;
  const int C_curr = imin(blockIdx.x * outputsPerBlock, m + n);
  const int C_next = imin(C_curr + outputsPerBlock, m + n);
  if (threadIdx.x == 0) {
    block_A_curr = co_rank(C_curr, A, m, B, n);
    block_A_next = co_rank(C_next, A, m, B, n);
  }
  __syncthreads();

  const int A_curr = block_A_curr;
  const int A_next = block_A_next;
  const int B_curr = C_curr - A_curr;
  const int B_next = C_next - A_next;
  const int A_length = A_next - A_curr;
  const int B_length = B_next - B_curr;
  const int C_length = C_next - C_curr;

  int C_completed = 0;
  int A_consumed = 0;
  int B_consumed = 0;
  while (C_completed < C_length) {
    const int A_S_length = imin(TILE, A_length - A_consumed);
    const int B_S_length = imin(TILE, B_length - B_consumed);
    for (int x = threadIdx.x; x < A_S_length; x += blockDim.x) {
      A_S[x] = A[A_curr + A_consumed + x];
    }
    for (int x = threadIdx.x; x < B_S_length; x += blockDim.x) {
      B_S[x] = B[B_curr + B_consumed + x];
    }
    __syncthreads();

    const int C_step = imin(TILE, C_length - C_completed);
    const int c_curr = imin(threadIdx.x * TILE_OUTPUTS_PER_THREAD, C_step);
    const int c_next = imin(c_curr + TILE_OUTPUTS_PER_THREAD, C_step);
    const int a_curr = co_rank(c_curr, A_S, A_S_length, B_S, B_S_length);
    const int a_next = co_rank(c_next, A_S, A_S_length, B_S, B_S_length);
    const int b_curr = c_curr - a_curr;
    const int b_next = c_next - a_next;
    merge_sequential(A_S + a_curr,
                     a_next - a_curr,
                     B_S + b_curr,
                     b_next - b_curr,
                     C + C_curr + C_completed + c_curr);

    const int A_step = co_rank(C_step, A_S, A_S_length, B_S, B_S_length);
    A_consumed += A_step;
    B_consumed += C_step - A_step;
    C_completed += C_step;
    __syncthreads();
  }
}

constexpr int TILE_MASK = TILE - 1;
static_assert((TILE & TILE_MASK) == 0, "TILE must be a power of two");

__host__ __device__ inline int circular_slot(int index) {
  return index & TILE_MASK;
}

__host__ __device__ inline int co_rank_circular(
    int k, const int* A_S, int A_S_start, int m, const int* B_S, int B_S_start, int n) {
  int i = imin(k, m);
  int j = k - i;
  int i_low = imax(0, k - n);
  int j_low = imax(0, k - m);
  bool searching = true;
  while (searching) {
    const bool i_too_large =
        i > 0 && j < n && A_S[circular_slot(A_S_start + i - 1)] > B_S[circular_slot(B_S_start + j)];
    const bool i_too_small =
        j > 0 && i < m &&
        B_S[circular_slot(B_S_start + j - 1)] >= A_S[circular_slot(A_S_start + i)];
    if (i_too_large) {
      const int delta = (i - i_low + 1) >> 1;
      j_low = j;
      i -= delta;
      j += delta;
    } else if (i_too_small) {
      const int delta = (j - j_low + 1) >> 1;
      i_low = i;
      i += delta;
      j -= delta;
    } else {
      searching = false;
    }
  }
  return i;
}

__host__ __device__ inline void merge_sequential_circular(
    const int* A_S, int A_S_start, int m, const int* B_S, int B_S_start, int n, int* C) {
  int i = 0;
  int j = 0;
  int k = 0;
  while (i < m && j < n) {
    const int a = A_S[circular_slot(A_S_start + i)];
    const int b = B_S[circular_slot(B_S_start + j)];
    if (a <= b) {
      C[k++] = a;
      ++i;
    } else {
      C[k++] = b;
      ++j;
    }
  }
  while (i < m) {
    C[k++] = A_S[circular_slot(A_S_start + i++)];
  }
  while (j < n) {
    C[k++] = B_S[circular_slot(B_S_start + j++)];
  }
}

__global__ void merge_circular_kernel(const int* A, int m, const int* B, int n, int* C) {
  __shared__ int A_S[TILE];
  __shared__ int B_S[TILE];
  __shared__ int block_A_curr;
  __shared__ int block_A_next;

  const int outputsPerBlock = (m + n + gridDim.x - 1) / gridDim.x;
  const int C_curr = imin(blockIdx.x * outputsPerBlock, m + n);
  const int C_next = imin(C_curr + outputsPerBlock, m + n);
  if (threadIdx.x == 0) {
    block_A_curr = co_rank(C_curr, A, m, B, n);
    block_A_next = co_rank(C_next, A, m, B, n);
  }
  __syncthreads();

  const int A_curr = block_A_curr;
  const int A_next = block_A_next;
  const int B_curr = C_curr - A_curr;
  const int B_next = C_next - A_next;
  const int A_length = A_next - A_curr;
  const int B_length = B_next - B_curr;
  const int C_length = C_next - C_curr;

  int C_completed = 0;
  int A_consumed = 0;
  int B_consumed = 0;
  int A_loaded = 0;
  int B_loaded = 0;
  while (C_completed < C_length) {
    const int A_window_end = imin(A_consumed + TILE, A_length);
    const int B_window_end = imin(B_consumed + TILE, B_length);
    for (int x = A_loaded + threadIdx.x; x < A_window_end; x += blockDim.x) {
      A_S[circular_slot(x)] = A[A_curr + x];
    }
    for (int x = B_loaded + threadIdx.x; x < B_window_end; x += blockDim.x) {
      B_S[circular_slot(x)] = B[B_curr + x];
    }
    A_loaded = A_window_end;
    B_loaded = B_window_end;
    __syncthreads();

    const int A_S_length = A_loaded - A_consumed;
    const int B_S_length = B_loaded - B_consumed;
    const int C_step = imin(TILE, C_length - C_completed);
    const int c_curr = imin(threadIdx.x * TILE_OUTPUTS_PER_THREAD, C_step);
    const int c_next = imin(c_curr + TILE_OUTPUTS_PER_THREAD, C_step);
    const int a_curr =
        co_rank_circular(c_curr, A_S, A_consumed, A_S_length, B_S, B_consumed, B_S_length);
    const int a_next =
        co_rank_circular(c_next, A_S, A_consumed, A_S_length, B_S, B_consumed, B_S_length);
    const int b_curr = c_curr - a_curr;
    const int b_next = c_next - a_next;
    merge_sequential_circular(A_S,
                              A_consumed + a_curr,
                              a_next - a_curr,
                              B_S,
                              B_consumed + b_curr,
                              b_next - b_curr,
                              C + C_curr + C_completed + c_curr);

    const int A_step =
        co_rank_circular(C_step, A_S, A_consumed, A_S_length, B_S, B_consumed, B_S_length);
    A_consumed += A_step;
    B_consumed += C_step - A_step;
    C_completed += C_step;
    __syncthreads();
  }
}

enum class MergeVariant { Basic, Tiled, Circular };

constexpr MergeVariant ALL_VARIANTS[] = {
    MergeVariant::Basic, MergeVariant::Tiled, MergeVariant::Circular};

static const char* variantName(MergeVariant variant) {
  switch (variant) {
    case MergeVariant::Basic:
      return "basic (global co-rank)";
    case MergeVariant::Tiled:
      return "tiled (shared memory)";
    case MergeVariant::Circular:
      return "circular buffer tiles";
  }
  return "unknown";
}

struct WorkSplit {
  int outputsPerThread;
  int outputsPerBlock;
};

constexpr int BASIC_THREADS_PER_BLOCK = 256;
constexpr WorkSplit DEFAULT_SPLIT = {64, 8 * TILE};
constexpr WorkSplit MANY_SMALL_SLICES_SPLIT = {5, 3 * TILE + 17};
constexpr int TIMED_OUTPUTS_PER_BLOCK = 16 * TILE;

constexpr unsigned RANDOM_SEED = 2024;
constexpr int FEW_DISTINCT_VALUES = 3;
constexpr int MANY_DISTINCT_VALUES = 1 << 30;
constexpr int LARGE_VALUE_RANGE = 1 << 26;
constexpr int LARGE_M = BY_BACKEND(1 << 24, 1 << 19);
constexpr int LARGE_N = LARGE_M + 12345;

constexpr int READS_AND_WRITES_PER_ELEMENT = 2;
constexpr double MS_PER_SECOND = 1e3;
constexpr double BYTES_PER_GB = 1e9;

struct DeviceMergeArrays {
  int* A;
  int* B;
  int* C;
  int m;
  int n;
};

static int* allocateDeviceInts(int count) {
  int* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, std::max(count, 1) * sizeof(int)));
  return pointer;
}

static DeviceMergeArrays allocateDeviceArrays(int m, int n) {
  DeviceMergeArrays arrays;
  arrays.A = allocateDeviceInts(m);
  arrays.B = allocateDeviceInts(n);
  arrays.C = allocateDeviceInts(m + n);
  arrays.m = m;
  arrays.n = n;
  return arrays;
}

static void freeDeviceArrays(DeviceMergeArrays& arrays) {
  CHECK(cudaFree(arrays.A));
  CHECK(cudaFree(arrays.B));
  CHECK(cudaFree(arrays.C));
}

static void copyToDevice(int* destination, const std::vector<int>& source) {
  if (!source.empty()) {
    CHECK(cudaMemcpy(
        destination, source.data(), source.size() * sizeof(int), cudaMemcpyHostToDevice));
  }
}

static void copyToHost(std::vector<int>& destination, const int* source) {
  if (!destination.empty()) {
    CHECK(cudaMemcpy(
        destination.data(), source, destination.size() * sizeof(int), cudaMemcpyDeviceToHost));
  }
}

static void launchBasicMerge(const DeviceMergeArrays& arrays, int outputsPerThread) {
  const int outputs = arrays.m + arrays.n;
  const unsigned threadsNeeded = cdiv(outputs, outputsPerThread);
  const unsigned blocks = cdiv(threadsNeeded, BASIC_THREADS_PER_BLOCK);
  launch(merge_basic_kernel,
         dim3(blocks),
         dim3(BASIC_THREADS_PER_BLOCK),
         arrays.A,
         arrays.m,
         arrays.B,
         arrays.n,
         arrays.C);
}

template <class TiledKernel>
static void launchTiledMerge(TiledKernel kernel,
                             const DeviceMergeArrays& arrays,
                             int outputsPerBlock) {
  const int outputs = arrays.m + arrays.n;
  const unsigned blocks = cdiv(outputs, outputsPerBlock);
  launch(kernel,
         dim3(blocks),
         dim3(TILED_THREADS_PER_BLOCK),
         arrays.A,
         arrays.m,
         arrays.B,
         arrays.n,
         arrays.C);
}

static void runMerge(MergeVariant variant, const DeviceMergeArrays& arrays, WorkSplit split) {
  if (arrays.m + arrays.n == 0) {
    return;
  }

  switch (variant) {
    case MergeVariant::Basic:
      launchBasicMerge(arrays, split.outputsPerThread);
      break;
    case MergeVariant::Tiled:
      launchTiledMerge(merge_tiled_kernel, arrays, split.outputsPerBlock);
      break;
    case MergeVariant::Circular:
      launchTiledMerge(merge_circular_kernel, arrays, split.outputsPerBlock);
      break;
  }
  CHECK(cudaGetLastError());
  CHECK(cudaDeviceSynchronize());
}

static std::vector<int> sortedRandom(int count, int maxValue, std::mt19937& rng) {
  std::uniform_int_distribution<int> distribution(0, maxValue);
  std::vector<int> values(count);
  for (int& value : values) {
    value = distribution(rng);
  }
  std::sort(values.begin(), values.end());
  return values;
}

static std::vector<int> mergeOnHost(const std::vector<int>& A, const std::vector<int>& B) {
  std::vector<int> C(A.size() + B.size());
  std::merge(A.begin(), A.end(), B.begin(), B.end(), C.begin());
  return C;
}

static std::vector<int> mergeOnDevice(MergeVariant variant,
                                      const std::vector<int>& A,
                                      const std::vector<int>& B,
                                      WorkSplit split) {
  DeviceMergeArrays arrays =
      allocateDeviceArrays(static_cast<int>(A.size()), static_cast<int>(B.size()));
  copyToDevice(arrays.A, A);
  copyToDevice(arrays.B, B);

  runMerge(variant, arrays, split);

  std::vector<int> C(A.size() + B.size());
  copyToHost(C, arrays.C);
  freeDeviceArrays(arrays);
  return C;
}

static const char* passOrFail(bool ok) {
  return ok ? "PASS" : "FAIL";
}

static bool checkCoRankWorkedExample() {
  const int A[] = {3, 5, 5, 8, 12};
  const int B[] = {1, 5, 6, 8, 9, 14};
  const int expectedCoRank[] = {0, 0, 1, 2, 3, 3, 3, 4, 4, 4, 5, 5};
  const int m = std::size(A);
  const int n = std::size(B);
  const int shownK = 6;

  bool ok = true;
  for (int k = 0; k <= m + n; ++k) {
    ok = ok && co_rank(k, A, m, B, n) == expectedCoRank[k];
  }

  const int shownI = co_rank(shownK, A, m, B, n);
  const int shownJ = shownK - shownI;
  std::printf("co_rank worked example (k = %d -> i = %d, j = %d): %s\n",
              shownK,
              shownI,
              shownJ,
              passOrFail(ok));
  return ok;
}

struct InputSizes {
  int m;
  int n;
};

static bool checkEdgeCases(std::mt19937& rng) {
  const InputSizes cases[] = {{0, 0},
                              {0, 7},
                              {9, 0},
                              {1, 1},
                              {5, 6},
                              {1000, 3},
                              {3, 1000},
                              {4097, 1},
                              {12345, 6789},
                              {20000, 20000}};
  const int valueRanges[] = {FEW_DISTINCT_VALUES, MANY_DISTINCT_VALUES};

  int passed = 0;
  int total = 0;
  for (const InputSizes& sizes : cases) {
    for (int maxValue : valueRanges) {
      const std::vector<int> A = sortedRandom(sizes.m, maxValue, rng);
      const std::vector<int> B = sortedRandom(sizes.n, maxValue, rng);
      const std::vector<int> expected = mergeOnHost(A, B);

      for (MergeVariant variant : ALL_VARIANTS) {
        const bool ok = mergeOnDevice(variant, A, B, MANY_SMALL_SLICES_SPLIT) == expected;
        passed += ok;
        ++total;
        if (!ok) {
          std::printf("  FAIL: %s with m=%d n=%d\n", variantName(variant), sizes.m, sizes.n);
        }
      }
    }
  }

  const bool allPassed = passed == total;
  std::printf("edge cases: %d/%d %s\n\n", passed, total, passOrFail(allPassed));
  return allPassed;
}

static double gigabytesPerSecond(int elements, double ms) {
  const double bytesMoved =
      static_cast<double>(elements) * READS_AND_WRITES_PER_ELEMENT * sizeof(int);
  const double seconds = ms / MS_PER_SECOND;
  return bytesMoved / seconds / BYTES_PER_GB;
}

static void printTimingRow(const char* label, double ms, int elements, const char* check) {
  std::printf("%-34s %10.2f %10.2f  %s\n", label, ms, gigabytesPerSecond(elements, ms), check);
}

struct TimedRun {
  MergeVariant variant;
  int outputsPerThread;
  const char* label;
};

static bool runTimedMerge(std::mt19937& rng) {
  const std::vector<int> A = sortedRandom(LARGE_M, LARGE_VALUE_RANGE, rng);
  const std::vector<int> B = sortedRandom(LARGE_N, LARGE_VALUE_RANGE, rng);
  const int outputs = LARGE_M + LARGE_N;

  std::vector<int> expected(outputs);
  Timer cpuTimer;
  std::merge(A.begin(), A.end(), B.begin(), B.end(), expected.begin());
  const double cpuMs = cpuTimer.ms();

  DeviceMergeArrays arrays = allocateDeviceArrays(LARGE_M, LARGE_N);
  copyToDevice(arrays.A, A);
  copyToDevice(arrays.B, B);
  runMerge(MergeVariant::Basic, arrays, DEFAULT_SPLIT);

  std::printf("merge of m = %d and n = %d ints (%d outputs)\n", LARGE_M, LARGE_N, outputs);
  std::printf("%-34s %10s %10s  %s\n", "variant", "time (ms)", "GB/s", "check");
  printTimingRow("std::merge (CPU, 1 thread)", cpuMs, outputs, "reference");

  const TimedRun runs[] = {{MergeVariant::Basic, 8, "basic, 8 outputs/thread"},
                           {MergeVariant::Basic, 64, "basic, 64 outputs/thread"},
                           {MergeVariant::Basic, 512, "basic, 512 outputs/thread"},
                           {MergeVariant::Tiled, 0, "tiled, 128 thr, TILE 1024"},
                           {MergeVariant::Circular, 0, "circular, 128 thr, TILE 1024"}};
  bool allPassed = true;
  std::vector<int> C(outputs);
  for (const TimedRun& run : runs) {
    CHECK(cudaMemset(arrays.C, 0, outputs * sizeof(int)));
    const WorkSplit split = {run.outputsPerThread, TIMED_OUTPUTS_PER_BLOCK};

    Timer gpuTimer;
    runMerge(run.variant, arrays, split);
    const double gpuMs = gpuTimer.ms();

    copyToHost(C, arrays.C);
    const bool ok = C == expected;
    allPassed = allPassed && ok;
    printTimingRow(run.label, gpuMs, outputs, passOrFail(ok));
  }
  freeDeviceArrays(arrays);

  std::printf("\n(GB/s counts one read and one write of every element.)\n");
  return allPassed;
}

int main() {
  std::printf("Parallel merge (chapter 13) on %s\n\n", CUDA_BACKEND);
  std::mt19937 rng(RANDOM_SEED);

  const bool workedExampleOk = checkCoRankWorkedExample();
  const bool edgeCasesOk = checkEdgeCases(rng);
  const bool timedMergeOk = runTimedMerge(rng);

  const bool allOk = workedExampleOk && edgeCasesOk && timedMergeOk;
  std::printf("%s\n", allOk ? "ALL PASS" : "SOME TESTS FAILED");
  return allOk ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. merge.cu -o merge && ./merge

## What you did

Every program above was checked against the CPU. Go back to any `%%writefile` cell, change something (a size, a block dimension, a constant) and run the pair again to see what changes.

The exercises of this topic, with solutions, are at the end of `ec2/11_merge/README.md`.

**Next:** open `ec2/12_sorting/colab_1_step_by_step.ipynb`.